# Harness Bench

**A very small benchmark to find the harness you need.**

Select the project's .venv Python kernel. Choose the experiment, then run the preparation and comparison cells below. Run All prepares the local services and starts the comparison. The progress bar advances once each trial finishes and shows the active harness, task, elapsed time and outcome.

In [ ]:
from pathlib import Path
import importlib
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "scripts/local_comparison.py").is_file())
sys.path.insert(0, str(root / "scripts"))
import local_comparison
importlib.reload(local_comparison)
from local_comparison import run_comparison, read_results

print("Python kernel", sys.executable)

## Choose the experiment

The demo has one task per harness. Smoke has two, quick has five, and full has eight. Each attempt repeats every selected task. Droid and DSH local routes await model-driven validation.

In [ ]:
model = "lfm3:1.7b"
harnesses = ["pi", "opencode", "droid", "dsh"]
suite = "demo"
attempts = 1

## Prepare Docker and Ollama

Start the Docker and Ollama applications. This cell builds the shared image if missing and downloads your selected model if needed. The first setup can take several minutes.

In [ ]:
import json
import subprocess
import urllib.request

subprocess.run(["docker", "info"], check=True, capture_output=True)
image = subprocess.run(
    ["docker", "image", "inspect", "harness-bench-base:1"],
    capture_output=True,
)
if image.returncode:
    subprocess.run(
        ["docker", "build", "-t", "harness-bench-base:1", "docker"],
        cwd=root, check=True,
    )

with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5) as response:
    installed_models = json.load(response)["models"]
if model not in {item["name"] for item in installed_models}:
    subprocess.run(["ollama", "pull", model], check=True)

print("Ready to compare", harnesses, "with", model)

## Run the comparison

Execute this cell to start. The bar counts completed trials, including failures. A trial includes CLI setup, model execution and grading. If it stops with an error, inspect the saved campaign logs. Interrupt the kernel to cancel a running trial.

In [ ]:
campaign = run_comparison(
    model=model,
    harnesses=harnesses,
    suite=suite,
    attempts=attempts,
)
print("Saved campaign", campaign)

## Read the results

Each row corresponds to one harness, task and attempt. Unknown usage fields remain empty. To inspect a previous campaign, replace campaign with its folder path before running this cell.

In [ ]:
from IPython.display import Markdown, display

rows = read_results(campaign)
display(Markdown((campaign / "report.md").read_text()))
print("CSV", campaign / "results.csv")

## Compare paired outcomes

Match trials by task and attempt. Review both output files when grades differ. Repeat the comparison before deciding whether a gap persists.

In [ ]:
pairs = {}
for row in rows:
    key = (row["task"], row["attempt"])
    outcome = row["reward"] if row["status"] == "graded" else row["status"]
    pairs.setdefault(key, {})[row["harness"]] = outcome
print("task | attempt | " + " | ".join(harnesses))
for (task, attempt), outcomes in sorted(pairs.items()):
    print(" | ".join([task, attempt] + [outcomes.get(h, "missing") for h in harnesses]))